# Task: Merge all raw data needed

**Unit of observation:** individual program beneficiary (`id_SSE`), the unit of randomization.
**Output:** `data/intermediate/` (person-level merged file + the person-month employment panel).

| Step | Source (`data/raw/`) | ID Key | Use |
|---|---|---|---|
| 1 | `RCT/worldbank/individ_randomization.dta` | `id` (= `id_SSE`) | Treatment arm, strata (municipality, age, program), socio-demographics |
| 2 | `RCT/worldbank/*crosswalk.dta` | `id_usuario` → `id_correo` → `id_SSE` | Link SkillLab users to beneficiaries |
| 3 | `RCT/ministry/*.dta` | `id_SSE` | Administrative outcomes (courses, formal employment, CV updates) |
| 4 | `RCT/skilllab/*.xlsx` | `user_id` | Take-up / usage; in-app survey |
| 5 | `RCT/opinaia/2025.08.06 Base.xlsx` | `Id_SSE` | Phone-survey outcomes |


No sample restrictions, recodes or outlier handling. 

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd
import openpyxl

pd.set_option("display.width", 200, "display.max_columns", 60)

# Set your paths 
# repo root = first parent folder that contains data/raw (works from code/cleaning/ or anywhere below the root)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw").is_dir())
RAW = ROOT / "data" / "raw"
RCT = RAW / "RCT"
INTERMEDIATE = ROOT / "data" / "intermediate"
INTERMEDIATE.mkdir(parents=True, exist_ok=True)

def read_dta(path):
    return pd.read_stata(path, convert_categoricals = False)

# Helper to assert uniqueness of the ID key
def assert_unique(df, key, name):
    dup = df[key].duplicated().sum()
    assert dup == 0, f"{name}: {dup} duplicated values of {key}"
    print(f"{name}: {len(df):,} rows, unique on {key}")

## 1. Randomization file 
One row per randomized beneficiary.  
`treat`: 0 = Control, 1 = T1 (CV), 2 = T2 (CV + Recs), 3 = T3 (CV + Recs + In-demand).  
`id Key:` the Ministry id (`id_SSE`)

In [6]:
# Load the randomization data 
rand = read_dta(RCT / "worldbank" / "individ_randomization.dta")
rand = rand.rename(columns = {"id": "id_SSE"})

# Assert that there are no duplicated individuals 
assert_unique(rand, "id_SSE", "randomization")
# Treatment groups
print(rand["treat"].value_counts(dropna = False).sort_index())
# NOTE: "grupo" indicates the program the beneficiary is enrolled in, i.e., fomentar or vat
print(rand["grupo"].value_counts(dropna = False))

randomization: 1,034,347 rows, unique on id_SSE
treat
0.0    258586
1.0    258587
2.0    258587
3.0    258587
Name: count, dtype: int64
grupo
fomentar    654417
vat         379930
Name: count, dtype: int64


## 2. Crosswalks (SkillLab user → e-mail id → Ministry id)
* `id_SSE_email_crosswalk.dta`: `id_correo` ↔ `id_SSE` (plus invitation/registration flags).
* `user_correo_id_crosswalk.dta`: SkillLab `id_usuario` ↔ `id_correo`.

NOTE: Any `id_correo` that maps to more than one `id_SSE` is treated as an **ambiguous link** and not used to attach SkillLab data.

In [7]:
# Crosswalks data
cw = read_dta(RCT / "worldbank" / "id_SSE_email_crosswalk.dta")
# User_correo IDs
uc = read_dta(RCT / "worldbank" / "user_correo_id_crosswalk.dta")
assert_unique(uc, "id_usuario", "user_correo crosswalk")

# Ambiguity diagnostics
n_sse_per_correo = cw.groupby("id_correo")["id_SSE"].nunique()
ambiguous_correo = n_sse_per_correo[n_sse_per_correo > 1].index
print("ambiguous id_correo (map to >1 id_SSE):", len(ambiguous_correo))
print("id_SSE with >1 id_correo:", (cw.groupby("id_SSE")["id_correo"].nunique() > 1).sum())

# Filter to only unambiguous id_correo, then create -> id_SSE link
link = (cw.loc[~cw["id_correo"].isin(ambiguous_correo), ["id_correo", "id_SSE"]]
          .drop_duplicates())
# Double check the filter
assert_unique(link, "id_correo", "unambiguous correo->SSE link")

# user_id -> id_SSE
user_link = uc.merge(link, on = "id_correo", how = "left", validate = "m:1")
print("SkillLab users linked to an id_SSE:", user_link["id_SSE"].notna().sum(), "of", len(user_link))

user_correo crosswalk: 1,385 rows, unique on id_usuario
ambiguous id_correo (map to >1 id_SSE): 11
id_SSE with >1 id_correo: 3
unambiguous correo->SSE link: 62,507 rows, unique on id_correo
SkillLab users linked to an id_SSE: 1279 of 1385


### 2b. Person-level invitation / registration information
Collapse the e-mail crosswalk to one row per `id_SSE` (first row after sorting; flags are max-combined) so it can be joined to the randomization data.

In [8]:
cw_cols = ["registered", "invited", "merge_OE_data", "merge_participant_data"]
cw_person = (cw.groupby("id_SSE")
               .agg(n_correo = ("id_correo", "nunique"),
                    registered = ("registered", "max"),
                    invited = ("invited", "max"),
                    correo_ambiguous = ("id_correo", lambda s: s.isin(ambiguous_correo).any()))
               .reset_index())
assert_unique(cw_person, "id_SSE", "crosswalk (person level)")
display(cw_person.head())

crosswalk (person level): 62,526 rows, unique on id_SSE


,id_SSE,n_correo,registered,invited,correo_ambiguous
0,28,1,0.0,1.0,False
1,34,1,0.0,1.0,False
2,36,1,0.0,1.0,False
3,50,1,0.0,1.0,False
4,53,1,0.0,1.0,False


## 3. Ministry administrative data
Files 1–4 and `receiving_benefits_variable.dta` each hold **one flag per id_SSE** (`enrolled_in_course`, `applied_to_portal_job`,
`allowed_companies_contact`, `updated_cv`, `receiving_benefit`). They are left-joined to the randomization data; an id absent from a file
gets `NaN` and a `<file>_in_file` indicator, so the coding of "absent" is decided in `clean.ipynb`, not here.

`soc_security_emp_status.dta` is a **person-month panel** of formal employment (2024-01 to 2025-07); it is stored separately as a long file
and linked back by `id_SSE`.

In [9]:
ministry_files = {
    "soc_security_data_file1": "enrolled_in_course",
    "soc_security_data_file2": "applied_to_portal_job",
    "soc_security_data_file3": "allowed_companies_contact",
    "soc_security_data_file4": "updated_cv",
    "receiving_benefits_variable": "receiving_benefit",
}

ministry = rand[["id_SSE"]].copy()
for fname, var in ministry_files.items():
    m = read_dta(RCT / "ministry" / f"{fname}.dta")
    assert_unique(m, "id_SSE", fname)
    m[f"{var}_in_file"] = 1
    n_before = len(ministry)
    ministry = ministry.merge(m, on = "id_SSE", how = "left", validate = "1:1")
    assert len(ministry) == n_before
    print(f"   {var}: {100 * m['id_SSE'].isin(rand['id_SSE']).mean():.1f}% of file ids are in randomization")
ministry.describe().T[["count", "mean"]]

soc_security_data_file1: 9,683 rows, unique on id_SSE
   enrolled_in_course: 100.0% of file ids are in randomization
soc_security_data_file2: 2,346 rows, unique on id_SSE
   applied_to_portal_job: 100.0% of file ids are in randomization
soc_security_data_file3: 8,720 rows, unique on id_SSE
   allowed_companies_contact: 100.0% of file ids are in randomization
soc_security_data_file4: 33,113 rows, unique on id_SSE
   updated_cv: 100.0% of file ids are in randomization
receiving_benefits_variable: 440,261 rows, unique on id_SSE
   receiving_benefit: 80.3% of file ids are in randomization


,count,mean
id_SSE,1034347.0,524212.244857
enrolled_in_course,9683.0,1.000000
enrolled_in_course_in_file,9683.0,1.000000
applied_to_portal_job,2346.0,1.000000
applied_to_portal_job_in_file,2346.0,1.000000
allowed_companies_contact,8720.0,1.000000
allowed_companies_contact_in_file,8720.0,1.000000
updated_cv,33113.0,1.000000
updated_cv_in_file,33113.0,1.000000
receiving_benefit,353481.0,1.000000


In [10]:
# Employment panel 
emp_panel = read_dta(RCT / "ministry" / "soc_security_emp_status.dta")
print(emp_panel.shape, "| persons:", emp_panel["id_SSE"].nunique(), "| months:", emp_panel["month"].min(), "-", emp_panel["month"].max())
print("share of panel ids in randomization:", emp_panel["id_SSE"].isin(rand["id_SSE"]).mean() * 100, "percent.")
emp_panel["in_randomization"] = emp_panel["id_SSE"].isin(rand["id_SSE"])
emp_panel.head()

(136344, 11) | persons: 6026 | months: 202401 - 202507
share of panel ids in randomization: 100.0 percent.


,id_SSE,Empleador,date_m,active_employment,month,Activo,iddepto,depto,idprovi,provi,month_str,in_randomization
0,2203,Empresa 1,2024-01-01,0.0,202401,NaN,,,,,2024-01,True
1,2203,Empresa 1,2024-02-01,0.0,202402,NaN,,,,,2024-02,True
2,2203,Empresa 1,2024-03-01,0.0,202403,NaN,,,,,2024-03,True
3,2203,Empresa 1,2024-04-01,0.0,202404,NaN,,,,,2024-04,True
4,2203,Empresa 1,2024-05-01,0.0,202405,NaN,,,,,2024-05,True


## 4. SkillLab platform data (take-up / usage)
Event-level sheets are aggregated to **one row per SkillLab `user_id`**, mapped to `id_SSE` through the crosswalks (section 2), then
aggregated to one row per `id_SSE`. The raw per-event tables are kept as intermediate files for the alignment measures in the PAP (4.1).

In [11]:
# Skilllab 
SL_FILE = RCT / "skilllab" / "28Oct2025_Raw_Data_Report_Argentina.xlsx"
sl = pd.read_excel(SL_FILE, sheet_name = None)
for k, v in sl.items():
    v.columns = [str(c).strip() for c in v.columns]
    if "title" in v:
        v["title"] = v["title"].astype(str).str.strip()
    print(f"{k:35s}", v.shape)

Sign-ups                            (3363, 5)
Skills                              (57245, 7)
CVs                                 (1423, 5)
Careers                             (3044, 5)
Courses                             (2550, 6)
Sessions                            (3457, 5)
Career Plan                         (2251, 13)
NPS & Satisfaction                  (222, 6)
Theory of Change Responses          (1419, 11)
Theory of Change Survey Questio     (6, 3)


In [12]:
signups   = sl["Sign-ups"]
sessions  = sl["Sessions"]
skills    = sl["Skills"]
cvs       = sl["CVs"]
careers   = sl["Careers"]
courses   = sl["Courses"]
career_plan = sl["Career Plan"]
toc       = sl["Theory of Change Responses"]   # in-app survey

assert_unique(signups, "user_id", "Sign-ups")

# --- one row per user ---------------------------------------------------------------
sl_user = signups[["user_id", "title", "date"]].rename(columns = {"title": "sl_portal", "date": "sl_signup_date"})
sl_user["sl_signup"] = 1

def count_by_user(df, name, mask = None):
    d = df if mask is None else df[mask]
    return d.groupby("user_id").size().rename(name)

parts = [
    count_by_user(sessions, "sl_n_sessions"),
    count_by_user(skills, "sl_n_skill_events"),
    skills.groupby("user_id")["skill_title"].nunique().rename("sl_n_unique_skills"),
    count_by_user(cvs, "sl_n_cv_exports"),
    count_by_user(careers, "sl_n_career_viewed", careers["metric"].eq("Career viewed")),
    count_by_user(careers, "sl_n_career_interest", careers["metric"].eq("Career interest")),
    count_by_user(courses, "sl_n_course_viewed", courses["metric"].eq("Course viewed")),
    count_by_user(courses, "sl_n_course_enrolled_click", courses["metric"].eq("Course enrolled")),
    count_by_user(career_plan, "sl_n_career_plans"),
]
sl_user = sl_user.merge(pd.concat(parts, axis=1).reset_index(), on="user_id", how="left")
count_cols = [c for c in sl_user.columns if c.startswith("sl_n_")]
sl_user[count_cols] = sl_user[count_cols].fillna(0)   # no event row == zero events among sign-ups

# in-app survey (Likert 1-5): keep every response row's phase, one row per user x phase, widen
toc_wide = (toc.sort_values("date")
               .groupby(["user_id", "phase"]).last().reset_index()
               .pivot(index="user_id", columns="phase",
                      values=["skill_awareness", "skill_confidence", "skill_communication",
                              "job_confidence", "interest_confidence", "career_awareness"]))
toc_wide.columns = [f"sl_toc_{a}_{b}" for a, b in toc_wide.columns]
sl_user = sl_user.merge(toc_wide.reset_index(), on="user_id", how="left")
assert_unique(sl_user, "user_id", "SkillLab user level")
sl_user.head()

Sign-ups: 3,363 rows, unique on user_id
SkillLab user level: 3,363 rows, unique on user_id


,user_id,sl_portal,sl_signup_date,sl_signup,sl_n_sessions,sl_n_skill_events,sl_n_unique_skills,sl_n_cv_exports,sl_n_career_viewed,sl_n_career_interest,sl_n_course_viewed,sl_n_course_enrolled_click,sl_n_career_plans,sl_toc_skill_awareness_final,sl_toc_skill_awareness_initial,sl_toc_skill_confidence_final,sl_toc_skill_confidence_initial,sl_toc_skill_communication_final,sl_toc_skill_communication_initial,sl_toc_job_confidence_final,sl_toc_job_confidence_initial,sl_toc_interest_confidence_final,sl_toc_interest_confidence_initial,sl_toc_career_awareness_final,sl_toc_career_awareness_initial
0,47296,Portal A -Fase 1,2025-04-30 11:15:00.590,1,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,47349,Portal A -Fase 1,2025-05-08 10:03:13.686,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,5.0,NaN,5.0,NaN,5.0,NaN,5.0,NaN,NaN,NaN,NaN
2,47418,Portal A -Fase 1,2025-05-14 12:58:43.542,1,1.0,11.0,11.0,1.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,47515,Portal A -Fase 1,2025-05-20 15:15:31.221,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,47518,Portal A -Fase 1,2025-05-20 15:27:13.768,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Attach id_SSE
sl_user = sl_user.merge(user_link.rename(columns = {"id_usuario": "user_id"}), on = "user_id", how = "left", validate = "1:1")
print("sign-ups with usable id_SSE link:", sl_user["id_SSE"].notna().sum(), "of", len(sl_user))

# user level -> person level (several SkillLab accounts can map to one beneficiary)
sum_cols = count_cols + ["sl_signup"]
other_cols = [c for c in sl_user.columns if c.startswith("sl_toc_")]
sl_person = (sl_user.dropna(subset=["id_SSE"])
                    .assign(id_SSE=lambda d: d["id_SSE"].astype("int64"))
                    .groupby("id_SSE")
                    .agg({**{c: "sum" for c in sum_cols}, **{c: "mean" for c in other_cols},
                          "sl_portal": "first", "sl_signup_date": "min", "user_id": "nunique"})
                    .rename(columns={"user_id": "sl_n_accounts"})
                    .reset_index())
assert_unique(sl_person, "id_SSE", "SkillLab person level")
sl_person["sl_signup"] = (sl_person["sl_signup"] > 0).astype(int)

sign-ups with usable id_SSE link: 1279 of 3363
SkillLab person level: 1,279 rows, unique on id_SSE


## 5. Opinaia phone survey (outcomes)
Sheet `Sheet1` holds the respondent-level data (the `Hoja1` sheet is only a pivot table). Question wording is in the questionnaire in `RCT/opinaia/`.

In [14]:
survey = pd.read_excel(RCT / "opinaia" / "2025.08.06 Base.xlsx", sheet_name = "Sheet1")
survey = survey.rename(columns={"Id_SSE": "id_SSE"})
assert_unique(survey, "id_SSE", "phone survey")
print(survey["Grupo"].value_counts())

# drop direct identifiers before anything is saved
survey = survey.drop(columns=["APELLIDO_NOMBRE", "TELEFONO", "Telefono_Normalizado"])
survey = survey.add_prefix("sv_").rename(columns={"sv_id_SSE": "id_SSE"})
print("survey ids found in randomization:", survey["id_SSE"].isin(rand["id_SSE"]).mean()*100, "percent.")

phone survey: 1,021 rows, unique on id_SSE
Grupo
T1 - SkillLab CV                  320
T2 - SkillLab Full                301
T3 - SkillLab Full + In-Demand    300
Control                           100
Name: count, dtype: int64
survey ids found in randomization: 100.0 percent.


## 6. Assemble the person-level file
Base = randomization file; everything else is left-joined on `id_SSE` with `validate = "1:1"` so the row count can never change.

In [15]:
merged = rand.copy()
n0 = len(merged)
for name, df in [("crosswalk", cw_person), ("ministry", ministry), ("skilllab", sl_person), ("survey", survey)]:
    merged = merged.merge(df, on = "id_SSE", how = "left", validate = "1:1")
    assert len(merged) == n0, f"row count changed after {name}"

merged["in_skilllab"] = merged["sl_signup"].notna().astype(int)
merged["in_survey"] = merged["sv_responseid"].notna().astype(int)
merged["in_emp_panel"] = merged["id_SSE"].isin(emp_panel["id_SSE"]).astype(int)
sl_cols = [c for c in merged.columns if c.startswith("sl_n_") or c == "sl_signup"]
print(merged.shape)

(1034347, 114)


### 6b. Merge diagnostics

In [16]:
checks = pd.DataFrame({
    "N": [n0, merged["in_skilllab"].sum(), merged["in_survey"].sum(), merged["in_emp_panel"].sum(),
          merged["enrolled_in_course_in_file"].sum(), merged["updated_cv_in_file"].sum()],
}, index = ["randomized", "in SkillLab data", "in phone survey", "in employment panel", "in ministry file1 (courses)", "in ministry file4 (CV)"])

display(checks)

print("\nCoverage by treatment arm (rows = arm):")
display(merged.groupby("treat")[["in_skilllab", "in_survey", "in_emp_panel"]].agg(["sum", "mean"]).round(4))
print("Survey 'Grupo' vs randomized treat (should line up 1:1):")
display(pd.crosstab(merged.loc[merged.in_survey == 1, "sv_Grupo"], merged.loc[merged.in_survey == 1, "treat"]))

,N
randomized,1034347.0
in SkillLab data,1279.0
in phone survey,1021.0
in employment panel,6026.0
in ministry file1 (courses),9683.0
in ministry file4 (CV),33113.0



Coverage by treatment arm (rows = arm):


in_skilllab         in_survey         in_emp_panel        
              sum    mean       sum    mean          sum    mean
treat                                                           
0.0             0  0.0000       100  0.0004         1472  0.0057
1.0           424  0.0016       320  0.0012         1527  0.0059
2.0           443  0.0017       301  0.0012         1523  0.0059
3.0           412  0.0016       300  0.0012         1504  0.0058

Survey 'Grupo' vs randomized treat (should line up 1:1):


treat,0.0,1.0,2.0,3.0
sv_Grupo,,,,
Control,100,0,0,0
T1 - SkillLab CV,0,320,0,0
T2 - SkillLab Full,0,0,301,0
T3 - SkillLab Full + In-Demand,0,0,0,300


## 7. Save to `data/intermediate/`

In [17]:
merged.to_csv(INTERMEDIATE / "merged_person.csv", index=False)
emp_panel.to_csv(INTERMEDIATE / "emp_panel_long.csv", index=False)

# event-level SkillLab tables (linked to id_SSE) for the alignment measures in PAP section 4.1
user_to_sse = user_link.rename(columns={"id_usuario": "user_id"})[["user_id", "id_SSE"]]
for name, df in [("skills", skills), ("careers", careers), ("courses", courses), ("cvs", cvs), ("sessions", sessions)]:
    df.merge(user_to_sse, on="user_id", how="left").to_csv(INTERMEDIATE / f"skilllab_{name}_events.csv", index=False)

sorted(p.name for p in INTERMEDIATE.iterdir())

['emp_panel_long.csv',
 'merged_person.csv',
 'skilllab_careers_events.csv',
 'skilllab_courses_events.csv',
 'skilllab_cvs_events.csv',
 'skilllab_sessions_events.csv',
 'skilllab_skills_events.csv']